# Attention Is All You Need

The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-to-German translation task, improving over the existing best results, including ensembles by over 2 BLEU. On the WMT 2014 English-to-French translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of the training costs of the best models from the literature. We show that the Transformer generalizes well to other tasks by applying it successfully to English constituency parsing both with large and limited training data.

---
*Notebook auto-generated by [visual-arxiv](https://github.com/visual-arxiv)*

## Setup
Install Manim and system dependencies for Colab.

In [ ]:
!pip install -q manim
!apt-get install -qq -y libcairo2-dev libpango1.0-dev ffmpeg > /dev/null 2>&1

## Animation Code
The Manim script below visualizes the core concepts of the paper.

In [ ]:
%%writefile animation.py
from manim import *

class PaperAnimation(Scene):
    def construct(self):
        # --- Title Scene ---
        title = Text("Attention Is All You Need").scale(1.5).to_edge(UP)
        subtitle = Text("The Transformer Architecture").scale(0.8).next_to(title, DOWN)
        self.play(Write(title))
        self.play(FadeIn(subtitle))
        self.wait(1.5)

        # --- Traditional Models Concept ---
        self.play(FadeOut(subtitle))
        traditional_concept_title = Text("Traditional Models: RNNs & CNNs with Attention", font_size=36).to_edge(UP)
        self.play(Transform(title, traditional_concept_title))
        
        encoder_block_trad = Rectangle(width=2.5, height=1.5, color=BLUE).shift(LEFT * 3 + DOWN * 1.5)
        encoder_text_trad = Text("Encoder (RNN/CNN)", font_size=24).move_to(encoder_block_trad.get_center())
        decoder_block_trad = Rectangle(width=2.5, height=1.5, color=GREEN).shift(RIGHT * 3 + DOWN * 1.5)
        decoder_text_trad = Text("Decoder (RNN/CNN)", font_size=24).move_to(decoder_block_trad.get_center())
        
        trad_enc_dec_arrow = Arrow(start=encoder_block_trad.get_right(), end=decoder_block_trad.get_left(), buff=0.2, color=WHITE)
        attention_mechanism_label = Text("Attention Mechanism", font_size=28, color=YELLOW).next_to(trad_enc_dec_arrow, UP, buff=0.3)
        attention_arrow_up = Arrow(start=encoder_block_trad.get_top(), end=attention_mechanism_label.get_bottom(), buff=0.1, color=YELLOW)
        attention_arrow_down = Arrow(start=attention_mechanism_label.get_bottom(), end=decoder_block_trad.get_top(), buff=0.1, color=YELLOW)

        self.play(Create(encoder_block_trad), Write(encoder_text_trad))
        self.play(Create(decoder_block_trad), Write(decoder_text_trad))
        self.play(Create(trad_enc_dec_arrow))
        self.play(FadeIn(attention_mechanism_label), Create(attention_arrow_up), Create(attention_arrow_down))
        self.wait(2)

        # --- Transition to Transformer: Dispensing with RNNs/CNNs ---
        dispensing_text = Text("Dispensing with Recurrence and Convolutions", font_size=36, color=RED).to_edge(UP)
        self.play(Transform(title, dispensing_text))
        self.wait(1)

        cross_out_rnn_cnn_1 = Line(start=encoder_text_trad.get_corner(UL), end=encoder_text_trad.get_corner(DR), color=RED, stroke_width=7)
        cross_out_rnn_cnn_2 = Line(start=decoder_text_trad.get_corner(UL), end=decoder_text_trad.get_corner(DR), color=RED, stroke_width=7)

        self.play(Create(cross_out_rnn_cnn_1), Create(cross_out_rnn_cnn_2))
        self.wait(1)

        self.play(FadeOut(encoder_text_trad), FadeOut(decoder_text_trad), FadeOut(cross_out_rnn_cnn_1), FadeOut(cross_out_rnn_cnn_2))
        self.play(FadeOut(trad_enc_dec_arrow, attention_arrow_up, attention_arrow_down))

        # --- Transformer: Solely Attention ---
        transformer_concept_title = Text("The Transformer: Based Solely on Attention", font_size=36).to_edge(UP)
        self.play(Transform(title, transformer_concept_title))

        # Move traditional attention label to a temporary position before fading it out fully
        self.play(attention_mechanism_label.animate.move_to(UP * 2).scale(0.8)) 
        self.play(FadeOut(encoder_block_trad), FadeOut(decoder_block_trad))

        input_seq = Text("Input Sequence", font_size=28, color=WHITE).shift(LEFT * 4 + UP * 0.5)
        output_seq = Text("Output Sequence", font_size=28, color=WHITE).shift(RIGHT * 4 + UP * 0.5)

        # --- Transformer Block Definitions (Revised for spacing and fit) ---
        # Encoder Block
        encoder_box = Rectangle(width=2.5, height=4.5, color=BLUE).shift(LEFT * 2) # Increased height
        encoder_label = Text("Encoder Stack", font_size=28).move_to(encoder_box.get_center() + UP * 1.8) # Adjusted position
        
        # Shapes for internal components
        self_att_enc_shape = Circle(radius=0.6, color=YELLOW, fill_opacity=0.6)
        ffn_enc_shape = Rectangle(width=1.2, height=0.5, color=ORANGE, fill_opacity=0.6)

        # Arrange internal components using VGroup for consistent spacing
        encoder_internal_components = VGroup(
            self_att_enc_shape,
            ffn_enc_shape
        ).arrange(DOWN, buff=0.3).move_to(encoder_box.get_center() + DOWN * 0.3) # Shifted down to balance with label

        self_att_enc = encoder_internal_components[0]
        ffn_enc = encoder_internal_components[1]

        # Text for internal components, scaled to fit
        self_att_enc_text = Text("Self-Attention", font_size=18).scale_to_fit_width(self_att_enc.get_width() * 0.8).move_to(self_att_enc.get_center())
        ffn_enc_text = Text("FFN", font_size=20).scale_to_fit_width(ffn_enc.get_width() * 0.8).move_to(ffn_enc.get_center())
        
        encoder_group = VGroup(encoder_box, encoder_label, self_att_enc, ffn_enc, self_att_enc_text, ffn_enc_text)

        # Decoder Block
        decoder_box = Rectangle(width=2.5, height=4.5, color=GREEN).shift(RIGHT * 2) # Increased height
        decoder_label = Text("Decoder Stack", font_size=28).move_to(decoder_box.get_center() + UP * 1.8) # Adjusted position

        # Shapes for internal components
        masked_self_att_dec_shape = Circle(radius=0.6, color=YELLOW, fill_opacity=0.6)
        enc_dec_att_shape = Circle(radius=0.6, color=PURPLE, fill_opacity=0.6)
        ffn_dec_shape = Rectangle(width=1.2, height=0.5, color=ORANGE, fill_opacity=0.6)

        # Arrange internal components using VGroup for consistent spacing
        decoder_internal_components = VGroup(
            masked_self_att_dec_shape,
            enc_dec_att_shape,
            ffn_dec_shape
        ).arrange(DOWN, buff=0.3).move_to(decoder_box.get_center() + DOWN * 0.3) # Shifted down to balance with label

        masked_self_att_dec = decoder_internal_components[0]
        enc_dec_att = decoder_internal_components[1]
        ffn_dec = decoder_internal_components[2]

        # Text for internal components, scaled to fit
        masked_self_att_dec_text = Text("Masked Self-Att", font_size=16).scale_to_fit_width(masked_self_att_dec.get_width() * 0.8).move_to(masked_self_att_dec.get_center())
        enc_dec_att_text = Text("Enc-Dec Att", font_size=16).scale_to_fit_width(enc_dec_att.get_width() * 0.8).move_to(enc_dec_att.get_center())
        ffn_dec_text = Text("FFN", font_size=20).scale_to_fit_width(ffn_dec.get_width() * 0.8).move_to(ffn_dec.get_center())
        
        decoder_group = VGroup(decoder_box, decoder_label, masked_self_att_dec, enc_dec_att, ffn_dec, masked_self_att_dec_text, enc_dec_att_text, ffn_dec_text)

        input_arrow = Arrow(start=input_seq.get_right(), end=encoder_box.get_left(), buff=0.1, color=WHITE)
        output_arrow = Arrow(start=decoder_box.get_right(), end=output_seq.get_left(), buff=0.1, color=WHITE)
        enc_dec_connection = Arrow(start=encoder_box.get_right(), end=enc_dec_att.get_left(), buff=0.1, color=RED)

        self.play(FadeIn(input_seq, output_seq))
        self.play(Create(encoder_group), Create(decoder_group))
        self.play(Create(input_arrow), Create(output_arrow), Create(enc_dec_connection))
        self.play(FadeOut(attention_mechanism_label)) # Fade out the old attention label completely
        self.wait(2)

        # --- Highlight Self-Attention ---
        self.play(Indicate(self_att_enc), Indicate(masked_self_att_dec), Indicate(enc_dec_att))
        self.wait(1.5)

        # --- Benefits ---
        self.play(FadeOut(input_seq, output_seq, input_arrow, output_arrow, enc_dec_connection, encoder_group, decoder_group))
        benefits_title = Text("Benefits of the Transformer", font_size=40).to_edge(UP)
        self.play(Transform(title, benefits_title))

        quality = Text("1. Superior Quality", font_size=36, color=WHITE).shift(UP * 1)
        parallel = Text("2. More Parallelizable", font_size=36, color=WHITE).shift(DOWN * 0)
        time = Text("3. Less Training Time", font_size=36, color=WHITE).shift(DOWN * 1)

        benefits_list = VGroup(quality, parallel, time).arrange(DOWN, buff=0.8)
        self.play(FadeIn(benefits_list))
        self.play(Indicate(quality), run_time=1)
        self.play(Indicate(parallel), run_time=1)
        self.play(Indicate(time), run_time=1)
        self.wait(2)

        # --- Final Conclusion ---
        final_title_text = Text("Attention Is All You Need", color=YELLOW).scale(1.5).to_edge(UP)
        final_model_text = Text("The Transformer", color=WHITE).scale(1).next_to(final_title_text, DOWN)
        self.play(FadeOut(benefits_list))
        self.play(Transform(title, final_title_text))
        self.play(FadeIn(final_model_text))
        self.wait(2)
        self.play(FadeOut(title, final_model_text))
        self.wait(1)

## Render the Animation

In [ ]:
!manim -ql animation.py PaperAnimation

## Display the Result

In [ ]:
import glob
from IPython.display import Video

videos = glob.glob('media/videos/animation/480p15/*.mp4')
if videos:
    display(Video(videos[0], embed=True))
else:
    print('No video found — check the render output above for errors.')